In [28]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu126
False


In [29]:
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [30]:
iris=load_iris()
df=pd.DataFrame(iris.data, columns=iris.feature_names)
df['target']=iris.target
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [31]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

In [32]:
x_train, y_train = train_df.drop(columns=['target']), train_df['target']
x_test, y_test = test_df.drop(columns=['target']), test_df['target']

In [33]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [34]:
x_train_tensor=torch.tensor(x_train, dtype=torch.float32)
x_test_tensor=torch.tensor(x_test, dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor=torch.tensor(y_test.values, dtype=torch.long)

In [35]:
class IrisClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

In [36]:
input_dim=x_train.shape[1]
hidden_dim=16
output_dim=3

In [37]:
model=IrisClassifier(input_dim, hidden_dim, output_dim)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

In [38]:
# training the model
num_epochs=1000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(x_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [50/1000], Loss: 0.9771
Epoch [100/1000], Loss: 0.6691
Epoch [150/1000], Loss: 0.3903
Epoch [200/1000], Loss: 0.2730
Epoch [250/1000], Loss: 0.1990
Epoch [300/1000], Loss: 0.1402
Epoch [350/1000], Loss: 0.0989
Epoch [400/1000], Loss: 0.0763
Epoch [450/1000], Loss: 0.0633
Epoch [500/1000], Loss: 0.0551
Epoch [550/1000], Loss: 0.0498
Epoch [600/1000], Loss: 0.0460
Epoch [650/1000], Loss: 0.0433
Epoch [700/1000], Loss: 0.0411
Epoch [750/1000], Loss: 0.0395
Epoch [800/1000], Loss: 0.0381
Epoch [850/1000], Loss: 0.0370
Epoch [900/1000], Loss: 0.0360
Epoch [950/1000], Loss: 0.0351
Epoch [1000/1000], Loss: 0.0342


In [41]:
model.eval()
with torch.no_grad():
    test_predictions = model(x_test_tensor)
    predicted_labels = torch.argmax(test_predictions, 1)
    # accuracy = (predicted_labels == y_test_tensor).float().mean()
    accuracy = (predicted_labels == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.9667


In [46]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [43]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    input_data=np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    input_data=scaler.transform(input_data)
    input_tensor=torch.tensor(input_data, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        prediction = model(input_tensor)
        predicted_label = torch.argmax(prediction, 1).item()
    return iris.target_names[predicted_label]

In [ ]:
predicted_class=predict_iris(5.1, 3.5, 1.4, 0.2)
print(f'Predicted Iris Species: {predicted_class}')

Predicted Iris Species: setosa


/media/thinkpad/NVME/kn_cv_course/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
